<a href="https://colab.research.google.com/github/kanezadjorda/UTS-BASIS-DATA-6C-BIS/blob/main/UTS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 pip install google-play-scraper


In [ ]:
!pip install google-play-scraper

import os
from google_play_scraper import reviews, Sort
import csv

app_id = 'id.qoin.korlantas.user'
result, _ = reviews(
    app_id,
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=200,
    filter_score_with=None
)

filename = 'ulasan_google_play.csv'
file_path = os.path.join(os.getcwd(), filename)

try:
    with open(file_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=['userName', 'score', 'at', 'content'])
        writer.writeheader()
        for review in result:
            writer.writerow({
                'userName': review['userName'],
                'score': review['score'],
                'at': review['at'],
                'content': review['content']
            })
    print(f"Berhasil menyimpan {len(result)} ulasan ke '{filename}'")
    if os.path.exists(file_path):
        print(f"Verifikasi: File '{filename}' berhasil dibuat di: {file_path}")
    else:
        print(f"Peringatan: File '{filename}' TIDAK DITEMUKAN setelah upaya penyimpanan!")
except Exception as e:
    print(f"Terjadi kesalahan saat menyimpan ulasan ke '{filename}': {e}")

In [ ]:
pip install transformers

In [ ]:
import pandas as pd
from transformers import pipeline

# Load the CSV file into a pandas DataFrame
df_reviews = pd.read_csv('ulasan_google_play.csv')

Sekarang, kita akan menginisialisasi pipeline untuk analisis sentimen. Karena ulasan dalam bahasa Indonesia, saya akan menggunakan model yang telah dilatih untuk bahasa Indonesia jika tersedia, atau model multilingual jika tidak.

In [ ]:
# Initialize a sentiment analysis pipeline
# Using a multilingual model that supports Indonesian
sentiment_pipeline = pipeline("sentiment-analysis", model="w11wo/indonesian-roberta-base-sentiment-classifier")

Selanjutnya, kita akan menerapkan analisis sentimen pada kolom 'content' dari DataFrame ulasan dan menampilkan hasilnya.

In [ ]:
print(f"DataFrame contains {len(df_reviews)} rows.")
print("Menampilkan 10 baris pertama untuk verifikasi:")
display(df_reviews.head(10))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- FIX: Apply sentiment analysis to create the 'sentiment' column ---
df_reviews['sentiment'] = df_reviews['content'].apply(lambda x: sentiment_pipeline(x)[0]['label'])
df_reviews['sentiment_score'] = df_reviews['content'].apply(lambda x: sentiment_pipeline(x)[0]['score'])
# ----------------------------------------------------------------------

# Calculate the distribution of sentiment labels
sentiment_counts = df_reviews['sentiment'].value_counts()

# Create a bar chart
plt.figure(figsize=(8, 6))
sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values, palette='viridis', hue=sentiment_counts.index, legend=False)
plt.title('Distribution of Sentiment Labels')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.show()